# GC2 — Final CIC-IDS2017 Data Pipeline

This notebook replaces the exploratory/repeated cells with one clean reproducible pipeline.

**Final experiment policy**
- Preserve all 15 labels in the clean master dataset.
- Hold out **Bot**, **FTP-Patator**, and **DoS Slowhttptest** as unknown attacks for the open-set experiment.
- Use the remaining **12 labels** for classifier train/validation/test splits (70/15/15).
- Autoencoder training and threshold validation use **BENIGN only** from the corresponding known splits.
- Retain small negative IAT values and `Init_Win_bytes_* = -1`; remove only the previously justified invalid negative duration/rate and structural header/segment rows.
- Save large datasets as **Parquet**, not CSV, to avoid disk-space problems.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_COL = "Label"
METADATA_COLS = ["source_file"]
UNKNOWN_CLASSES = ["Bot", "FTP-Patator", "DoS Slowhttptest"]
RARE_CLASSES = ["Heartbleed", "Web Attack - SQL Injection", "Infiltration"]

RAW_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed/Data_V2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "Monday": RAW_DIR / "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday": RAW_DIR / "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday": RAW_DIR / "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday_AM": RAW_DIR / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday_PM": RAW_DIR / "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday_AM": RAW_DIR / "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday_PortScan": RAW_DIR / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday_DDoS": RAW_DIR / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
}

print("Output directory:", OUTPUT_DIR.resolve())


Output directory: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\Data_V2


## 1. Load, standardize and combine the 8 raw CIC-IDS2017 files


In [2]:
dfs = []

label_mapping = {
    "Web Attack � Brute Force": "Web Attack - Brute Force",
    "Web Attack � Sql Injection": "Web Attack - SQL Injection",
    "Web Attack � XSS": "Web Attack - XSS",
}

for source_name, path in files.items():
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = frame.columns.str.strip()
    frame[TARGET_COL] = frame[TARGET_COL].astype(str).str.strip().replace(label_mapping)
    frame["source_file"] = source_name
    dfs.append(frame)
    print(f"{source_name:16} {frame.shape}")

data = pd.concat(dfs, ignore_index=True)
del dfs

print("\nCombined shape:", data.shape)
print("Classes:", data[TARGET_COL].nunique())
assert len(data) == 2_830_743
assert data[TARGET_COL].nunique() == 15


FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\data\\raw\\Monday-WorkingHours.pcap_ISCX.csv'

## 2. Remove NaN / ±Inf rows
The earlier audit showed 2,867 unique affected rows (~0.1013%); rare classes were not materially damaged by removal.


In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns
invalid_numeric_mask = data.isna().any(axis=1) | np.isinf(data[numeric_cols]).any(axis=1)
print("NaN/Inf rows removed:", int(invalid_numeric_mask.sum()))

df_clean = data.loc[~invalid_numeric_mask].copy()
del data
df_clean.reset_index(drop=True, inplace=True)

assert len(df_clean) == 2_827_876
assert df_clean.isna().sum().sum() == 0
numeric_cols = df_clean.select_dtypes(include=np.number).columns
assert np.isinf(df_clean[numeric_cols]).sum().sum() == 0
print("Shape:", df_clean.shape)


NaN/Inf rows removed: 2867
Shape: (2827876, 80)


## 3. Remove invalid negative duration/rate flows
115 unique BENIGN flows had negative `Flow Duration`; all also had negative `Flow Packets/s`, and 85 had negative `Flow Bytes/s`.


In [ ]:
negative_duration_rate_mask = (
    (df_clean["Flow Duration"] < 0) |
    (df_clean["Flow Bytes/s"] < 0) |
    (df_clean["Flow Packets/s"] < 0)
)

print("Invalid duration/rate rows removed:", int(negative_duration_rate_mask.sum()))
df_clean = df_clean.loc[~negative_duration_rate_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
assert len(df_clean) == 2_827_761


Invalid duration/rate rows removed: 115


## 4. Resolve conflicting labels and duplicate ML feature vectors
Rows with identical model features but different labels are removed entirely; remaining repeated feature vectors keep one representative. `source_file` is metadata and is ignored for modelling duplicates.


In [ ]:
feature_cols_pre = [c for c in df_clean.columns if c not in [TARGET_COL] + METADATA_COLS]

# Hash feature vectors to avoid a very large multi-column groupby/index.
feature_hash = pd.util.hash_pandas_object(df_clean[feature_cols_pre], index=False)
label_counts = df_clean.groupby(feature_hash, sort=False)[TARGET_COL].nunique()
conflicting_hashes = label_counts[label_counts > 1].index
conflict_mask = feature_hash.isin(conflicting_hashes)

print("Conflicting feature patterns:", len(conflicting_hashes))
print("Rows removed for label conflict:", int(conflict_mask.sum()))

df_clean = df_clean.loc[~conflict_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
assert len(df_clean) == 2_821_095

# Recompute hash after conflict removal and keep one row per feature vector.
feature_hash = pd.util.hash_pandas_object(df_clean[feature_cols_pre], index=False)
duplicate_mask = feature_hash.duplicated(keep="first")
print("Ordinary duplicate feature vectors removed:", int(duplicate_mask.sum()))

df_clean = df_clean.loc[~duplicate_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
assert len(df_clean) == 2_519_297


Conflicting feature patterns: 697
Rows removed for label conflict: 6666
Ordinary duplicate feature vectors removed: 301798


## 5. Remove redundant and constant columns


In [ ]:
constant_cols = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
]

cols_to_drop = ["Fwd Header Length.1"] + constant_cols
df_clean.drop(columns=cols_to_drop, inplace=True, errors="raise")
print("Shape after column cleanup:", df_clean.shape)
assert df_clean.shape[1] == 71


Shape after column cleanup: (2519297, 71)


## 6. Remove 35 structurally invalid negative header/segment rows
These 35 BENIGN flows contained extremely large negative header/segment measurements and were removed.


In [ ]:
invalid_structure_mask = (
    (df_clean["Fwd Header Length"] < 0) |
    (df_clean["Bwd Header Length"] < 0) |
    (df_clean["min_seg_size_forward"] < 0)
)

print("Invalid structural rows removed:", int(invalid_structure_mask.sum()))
df_clean = df_clean.loc[~invalid_structure_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
assert len(df_clean) == 2_519_262
assert df_clean.shape == (2_519_262, 71)


Invalid structural rows removed: 35


## 7. Document retained negative values

These are **intentionally retained**:
- `Init_Win_bytes_forward = -1` and `Init_Win_bytes_backward = -1` (special/sentinel-like values occurring in a large fraction of records).
- Small negative `Flow IAT Min` and `Fwd IAT Min` values. Removing them would disproportionately affect rare attacks (including Heartbleed).

Do not apply a blanket `X >= 0` rule.


In [ ]:
retained_negative_cols = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "Flow IAT Min",
    "Fwd IAT Min",
]

for col in retained_negative_cols:
    mask = df_clean[col] < 0
    print(f"{col:26} negatives={int(mask.sum()):,} min={df_clean.loc[mask, col].min() if mask.any() else None}")


Init_Win_bytes_forward     negatives=910,977 min=-1
Init_Win_bytes_backward    negatives=1,215,330 min=-1
Flow IAT Min               negatives=2,773 min=-14
Fwd IAT Min                negatives=17 min=-12


## 8. Final clean-master validation


In [ ]:
FEATURE_COLS = [c for c in df_clean.columns if c not in [TARGET_COL] + METADATA_COLS]
numeric_cols = df_clean.select_dtypes(include=np.number).columns
feature_hash = pd.util.hash_pandas_object(df_clean[FEATURE_COLS], index=False)

print("Shape:", df_clean.shape)
print("Classes:", df_clean[TARGET_COL].nunique())
print("ML features:", len(FEATURE_COLS))
print("NaN:", int(df_clean.isna().sum().sum()))
print("Inf:", int(np.isinf(df_clean[numeric_cols]).sum().sum()))
print("Duplicate feature vectors:", int(feature_hash.duplicated().sum()))
print("\nClass distribution:", df_clean[TARGET_COL].value_counts())

assert df_clean.shape == (2_519_262, 71)
assert len(FEATURE_COLS) == 69
assert df_clean[TARGET_COL].nunique() == 15
assert df_clean.isna().sum().sum() == 0
assert np.isinf(df_clean[numeric_cols]).sum().sum() == 0
assert feature_hash.duplicated().sum() == 0


Shape: (2519262, 71)
Classes: 15
ML features: 69
NaN: 0
Inf: 0
Duplicate feature vectors: 0

Class distribution: Label
BENIGN                        2094218
DoS Hulk                       172717
DDoS                           128011
PortScan                        90130
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5384
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - SQL Injection         21
Heartbleed                         11
Name: count, dtype: int64


## 9. Save the frozen clean master
Large files are saved as Parquet only.


In [ ]:
MASTER_PATH = OUTPUT_DIR / "cicids2017_clean_master_final.parquet"
df_clean.to_parquet(MASTER_PATH, index=False)
print("Saved:", MASTER_PATH.resolve())


Saved: /Users/anaghamohan/Desktop/GC2/Hybrid-Network-Intrusion-Detection/notebooks/Data_V2/cicids2017_clean_master_final.parquet


## 10. Create the final 3-class unknown set and 12-class known pool


In [ ]:
unknown_mask = df_clean[TARGET_COL].isin(UNKNOWN_CLASSES)
unknown_data = df_clean.loc[unknown_mask].copy()
known_pool = df_clean.loc[~unknown_mask].copy()

print("Full master:", len(df_clean))
print("Known development pool:", len(known_pool))
print("Unknown test data:", len(unknown_data))
print("\nUnknown distribution:", unknown_data[TARGET_COL].value_counts())

assert len(unknown_data) == 13_107
assert len(known_pool) == 2_506_155
assert known_pool[TARGET_COL].isin(UNKNOWN_CLASSES).sum() == 0
assert known_pool[TARGET_COL].nunique() == 12


Full master: 2519262
Known development pool: 2506155
Unknown test data: 13107

Unknown distribution: Label
FTP-Patator         5931
DoS Slowhttptest    5228
Bot                 1948
Name: count, dtype: int64


## 11. Split the 12 known classes: 70% train / 15% validation / 15% test


In [ ]:
known_train, known_temp = train_test_split(
    known_pool,
    test_size=0.30,
    stratify=known_pool[TARGET_COL],
    random_state=RANDOM_STATE,
)

known_val, known_test = train_test_split(
    known_temp,
    test_size=0.50,
    stratify=known_temp[TARGET_COL],
    random_state=RANDOM_STATE,
)

del known_temp, known_pool

print("Known train:", known_train.shape)
print("Known validation:", known_val.shape)
print("Known test:", known_test.shape)

for name, frame in {"train": known_train, "validation": known_val, "test": known_test}.items():
    print(name, "classes=", frame[TARGET_COL].nunique(), "unknown leakage=", int(frame[TARGET_COL].isin(UNKNOWN_CLASSES).sum()))
    assert frame[TARGET_COL].nunique() == 12
    assert frame[TARGET_COL].isin(UNKNOWN_CLASSES).sum() == 0


Known train: (1754308, 71)
Known validation: (375923, 71)
Known test: (375924, 71)
train classes= 12 unknown leakage= 0
validation classes= 12 unknown leakage= 0
test classes= 12 unknown leakage= 0


## 12. Autoencoder handoff definition
Do not save duplicate multi-GB benign files unless Member 3 explicitly needs them. They can be derived from the known splits.


In [ ]:
ae_train_benign = known_train.loc[known_train[TARGET_COL] == "BENIGN"]
ae_val_benign = known_val.loc[known_val[TARGET_COL] == "BENIGN"]

print("AE BENIGN train:", ae_train_benign.shape)
print("AE BENIGN validation:", ae_val_benign.shape)
print("Unknown evaluation:", unknown_data.shape)

assert ae_train_benign[TARGET_COL].eq("BENIGN").all()
assert ae_val_benign[TARGET_COL].eq("BENIGN").all()

print("\nMember 3 rule: fit preprocessing/scaler and AE using BENIGN training data only; use BENIGN validation for threshold selection; use the 3 held-out attacks only for final unknown evaluation / KB testing.")


AE BENIGN train: (1465952, 71)
AE BENIGN validation: (314133, 71)
Unknown evaluation: (13107, 71)

Member 3 rule: fit preprocessing/scaler and AE using BENIGN training data only; use BENIGN validation for threshold selection; use the 3 held-out attacks only for final unknown evaluation / KB testing.


## 13. Save only the essential derived datasets


In [ ]:
paths = {
    "known_train": OUTPUT_DIR / "known_train.parquet",
    "known_validation": OUTPUT_DIR / "known_validation.parquet",
    "known_test": OUTPUT_DIR / "known_test.parquet",
    "unknown_test": OUTPUT_DIR / "unknown_test_Bot_FTP_Slowhttptest.parquet",
}

known_train.to_parquet(paths["known_train"], index=False)
known_val.to_parquet(paths["known_validation"], index=False)
known_test.to_parquet(paths["known_test"], index=False)
unknown_data.to_parquet(paths["unknown_test"], index=False)

for name, path in paths.items():
    print(name, "->", path.resolve(), f"({path.stat().st_size / 1024**2:.1f} MB)")


known_train -> /Users/anaghamohan/Desktop/GC2/Hybrid-Network-Intrusion-Detection/notebooks/Data_V2/known_train.parquet (245.5 MB)
known_validation -> /Users/anaghamohan/Desktop/GC2/Hybrid-Network-Intrusion-Detection/notebooks/Data_V2/known_validation.parquet (53.3 MB)
known_test -> /Users/anaghamohan/Desktop/GC2/Hybrid-Network-Intrusion-Detection/notebooks/Data_V2/known_test.parquet (53.2 MB)
unknown_test -> /Users/anaghamohan/Desktop/GC2/Hybrid-Network-Intrusion-Detection/notebooks/Data_V2/unknown_test_Bot_FTP_Slowhttptest.parquet (1.3 MB)


## 14. Save lightweight metadata / manifest


In [ ]:
pd.DataFrame({"feature": FEATURE_COLS}).to_csv(OUTPUT_DIR / "feature_list.csv", index=False)

manifest = pd.DataFrame([
    {"dataset": "clean_master", "rows": len(df_clean), "classes": df_clean[TARGET_COL].nunique()},
    {"dataset": "known_train", "rows": len(known_train), "classes": known_train[TARGET_COL].nunique()},
    {"dataset": "known_validation", "rows": len(known_val), "classes": known_val[TARGET_COL].nunique()},
    {"dataset": "known_test", "rows": len(known_test), "classes": known_test[TARGET_COL].nunique()},
    {"dataset": "unknown_test", "rows": len(unknown_data), "classes": unknown_data[TARGET_COL].nunique()},
    {"dataset": "ae_train_benign_view", "rows": len(ae_train_benign), "classes": 1},
    {"dataset": "ae_val_benign_view", "rows": len(ae_val_benign), "classes": 1},
])
manifest.to_csv(OUTPUT_DIR / "dataset_manifest.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "unknown_classes": UNKNOWN_CLASSES,
    "rare_classes": RARE_CLASSES,
    "target_column": TARGET_COL,
    "metadata_columns": METADATA_COLS,
    "ml_feature_count": len(FEATURE_COLS),
    "retained_negative_policy": {
        "Init_Win_bytes_forward": "retain -1",
        "Init_Win_bytes_backward": "retain -1",
        "Flow IAT Min": "retain small negatives",
        "Fwd IAT Min": "retain small negatives",
    },
}
with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(manifest)


                dataset     rows  classes
0          clean_master  2519262       15
1           known_train  1754308       12
2      known_validation   375923       12
3            known_test   375924       12
4          unknown_test    13107        3
5  ae_train_benign_view  1465952        1
6    ae_val_benign_view   314133        1


## 15. Final freeze check


In [ ]:
print("========== FINAL DATA FREEZE ==========")
print("Master:", df_clean.shape)
print("Features:", len(FEATURE_COLS))
print("Unknown classes:", UNKNOWN_CLASSES)
print("Unknown rows:", len(unknown_data))
print("Unknown distribution:", unknown_data[TARGET_COL].value_counts())
print("Train / Val / Test:", len(known_train), len(known_val), len(known_test))
print("Unknown leakage:",
      int(known_train[TARGET_COL].isin(UNKNOWN_CLASSES).sum()),
      int(known_val[TARGET_COL].isin(UNKNOWN_CLASSES).sum()),
      int(known_test[TARGET_COL].isin(UNKNOWN_CLASSES).sum()))
print("AE train labels:", ae_train_benign[TARGET_COL].unique())
print("AE val labels:", ae_val_benign[TARGET_COL].unique())
print("\nDATA PIPELINE FROZEN.")


========== FINAL DATA FREEZE ==========
Master: (2519262, 71)
Features: 69
Unknown classes: ['Bot', 'FTP-Patator', 'DoS Slowhttptest']
Unknown rows: 13107
Unknown distribution: Label
FTP-Patator         5931
DoS Slowhttptest    5228
Bot                 1948
Name: count, dtype: int64
Train / Val / Test: 1754308 375923 375924
Unknown leakage: 0 0 0
AE train labels: ['BENIGN']
AE val labels: ['BENIGN']

DATA PIPELINE FROZEN.


## Data for Autoencoder

- run this block of code to get benign csv from classifier

In [ ]:
feature_cols = [
    col for col in ae_train_benign.columns
    if col not in ["Label", "source_file"]
]



print("AE training:", ae_train_benign.shape)
print("AE validation:", ae_val_benign.shape)
print("Number of features:", len(feature_cols))

ae_train_benign.to_parquet(
    "../../data/processed/Data_V2/autoencoder_benign_train.parquet",
    index=False
)

ae_val_benign.to_parquet(
    "../../data/processed/Data_V2/autoencoder_benign_validation.parquet",
    index=False
)

AE training: (1465952, 71)
AE validation: (314133, 71)
Number of features: 69


## To Convert parquet to csv

- run this

In [ ]:
def convert(parquet_file):
    csv_file = parquet_file.with_suffix(".csv")

    print(f"Converting: {parquet_file.name}")

    df = pd.read_parquet(parquet_file)

    df.to_csv(
        csv_file,
        index=False
    )
    print(f"Saved: {csv_file.name}")
    print(f"Shape: {df.shape}\n")

    del df

## call function

#convert(file_name)